# Exact Linearization of Integer × Continuous — Before, Principle, Application, and Effect

When a product of "integer quantity × continuous quantity" (e.g., number of batches × batch size, number of machines × uptime) enters a constraint, SCIP generally treats it as a bilinear term using **McCormick relaxation**. Because information on the integer side is discarded in the relaxation, the lower bound becomes loose and the gap struggles to close.

This notebook tracks this phenomenon and its remedy through the following consistent pattern:

1. **Issue (before)** — Diagnose the model as is (bilinear) with `mk.analyze` and confirm `weak_relaxation`
2. **Principle (principle)** — See the gap of the McCormick envelope visually
3. **Application (how)** — Apply `mk.linearize_product` in a single line
4. **Effect (before/after)** — Compare the root dual bound, gap, and node count using `mk.compare_variants`

The subject is **Batch Reactor Scheduling** (`samples/others/scheduling_plant.py`).
The product of batch count `n` (integer) × batch size `s` (continuous) enters the triple products for demand satisfaction and energy.

In [ ]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Issue (before) — Diagnosing the baseline

We run `mk.analyze` on the model left as bilinear (`linearize_ns=False`).
`analyze` actually solves the model, collects dual bound stalling, spatial branching bias, and nonlinear constraint violations, and returns the symptoms in order of severity.

In [ ]:
report = mk.analyze(lambda: sp.build_model(linearize_ns=False),
                    name="plant-baseline", time_limit=10)
print(report.summary())
print("\n支配ボトルネック:", report.metrics.get("bottleneck_type"),
      "/ 空間分枝の寄与:", f"{report.metrics.get('spatial_share', 0):.1%}")

The triple product $n\cdot s\cdot(T-T_0)$ (energy constraint) and others are cited as bottlenecks, triggering `weak_relaxation` (weak relaxation). The recipe is "Use `mk.linearize_product` for integer × continuous."
Next, we confirm the rationale for this visually.

## 2. Principle (principle) — Gap of the McCormick Envelope

Consider the product $w = ns$ of an integer $n$ and continuous $s$. SCIP **continuously relaxes** $n$ and applies a McCormick envelope to the bilinear term. Thus, in the $(s, w)$ plane, **all** of the band region from the line $w=s$ at $n=1$ to the line $w=3s$ at $n=3$ is permitted by the relaxation.
However, the true feasible set for the integer consists of **only 3 lines**: $w=1\cdot s,\ 2\cdot s,\ 3\cdot s$.
The interior of the band (shaded in the figure below) is the relaxation gap = the room where the lower bound is loose.

In [ ]:
# n in {1,2,3}, s in [0,10] で w=n*s を描く(小さい値域で図を見やすく)
s = np.linspace(0, 10, 100)
fig = go.Figure(layout=base_layout(
    "整数×連続の積 w = n·s : 緩和が許す領域 vs 真の実行可能直線",
    "連続量 s", "積 w = n·s", height=420))
# McCormick(n を連続緩和)が許す帯: s <= w <= 3s
fig.add_trace(go.Scatter(x=np.r_[s, s[::-1]], y=np.r_[3*s, s[::-1]],
    fill="toself", fillcolor="rgba(194,90,0,0.12)", line=dict(width=0),
    name="緩和が許す領域 (n を連続扱い)", hoverinfo="skip"))
# 真の整数直線 n=1,2,3
for v, col in zip([1, 2, 3], [C["s1"], C["s2"], C["ink"]]):
    fig.add_trace(go.Scatter(x=s, y=v*s, mode="lines",
        line=dict(color=col, width=2.5), name=f"n={v} (真の値 w={v}·s)",
        hovertemplate="n=" + str(v) + ": w=%{y:.1f}<extra></extra>"))
# ギャップの可視化: s=8 での縦幅
fig.add_annotation(x=8, y=(1*8+3*8)/2, ax=8, ay=1*8,
    text="緩和ギャップ", showarrow=True, arrowhead=2, arrowcolor=C["warn"],
    font=dict(color=C["warn"], size=12), xshift=40)
fig.add_shape(type="line", x0=8, y0=1*8, x1=8, y1=3*8,
    line=dict(color=C["warn"], width=1.5, dash="dot"))
show(fig)

**Exact linearization eliminates this band.** By creating an indicator variable $\delta_v$ (which is 1 when $n=v$) for each value $v$ of $n$, decomposing $s$ into $s = \sum_v s_v$ (where $s_v$ is valid only when $\delta_v=1$), and writing $w = \sum_v v\,s_v$, the feasible set of the relaxation perfectly matches the **convex hull of the true 3 lines**. In principle, the gap that "fills the spaces between integers" like the McCormick band disappears (an exact transformation with 0 relaxation gap).

## 3. Application (how) — `mk.linearize_product`

You can exactly linearize the product by just calling a general-purpose helper in a single line. `scheduling_plant.build_model(linearize_ns=True)` replaces `n·s` for each job with this single line.

```python
ns = mk.linearize_product(m, n, s, y_lb=1, y_ub=N_MAX, x_lb=S_MIN, x_ub=S_MAX, name="ns")
# From here on, ns is the exact linear representation of n*s. Products like ns*X or ns*(T-T0) drop to bilinear
```

Below is a minimal working check. We solve `n·s ≥ 12` as an exact linear constraint.

In [ ]:
from pyscipopt import Model
m = Model(); m.hideOutput()
n = m.addVar(vtype="I", lb=1, ub=3, name="n")
s = m.addVar(lb=0.0, ub=10.0, name="s")
ns = mk.linearize_product(m, n, s, 1, 3, 0.0, 10.0, "ns")
m.addCons(ns >= 12)
m.setObjective(n + s, "minimize")
m.optimize()
print(f"n={m.getVal(n):.0f}  s={m.getVal(s):.2f}  n*s={m.getVal(ns):.2f}  (>= 12 を厳密に満たす)")

## 4. Effect (before/after) — `mk.compare_variants`

We solve the bilinear (baseline) and exact linearization (linearized) models under the same time limit, and compare the **root dual bound, final gap, and node count**. The true way to measure the quality of a formulation is by the **root dual bound**, which is not confounded by search dynamics (gap/nodes are just for reference).

In [ ]:
df = mk.compare_variants(
    {"baseline(bilinear)": lambda: sp.build_model(linearize_ns=False),
     "linearized":         lambda: sp.build_model(linearize_ns=True)},
    time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
base = df.loc[df["variant"] == "baseline(bilinear)"].iloc[0]
lin  = df.loc[df["variant"] == "linearized"].iloc[0]

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
labels = ["baseline", "linearized"]
bar_colors = [C["muted"], C["s1"]]
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], lin["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["final_gap"]*100, lin["final_gap"]*100], lambda v: f"{v:.0f}%")
add_bars(3, [base["nodes"], lin["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="厳密線形化の before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [ ]:
print(f"ルート双対境界 : {base['root_dual']:.1f} -> {lin['root_dual']:.1f} "
      f"(+{(lin['root_dual']/base['root_dual']-1)*100:.0f}%)")
print(f"最終gap        : {base['final_gap']:.1%} -> {lin['final_gap']:.1%}")
print(f"ノード数       : {int(base['nodes'])} -> {int(lin['nodes'])}")

## Summary

- **The root dual bound significantly increases**. The gap of the McCormick band disappears, and the lower bound of the minimization problem is tight from the beginning. Due to the tightened lower bound, both the final gap and the number of nodes shrink.
- The optimal value itself does not change (an equivalent transformation). It is an improvement that **only tightens the relaxation**.

### Why SCIP Doesn't Do It Automatically

SCIP applies a general-purpose McCormick relaxation to bilinear terms. A **problem-structure-dependent reformulation** like "decomposing the integer side with indicator variables makes it exact" cannot be automatically detected from an already built model (it is outside the scope of SCIP's presolve). Therefore, the division of labor is that the diagnosis points out the structure, and the modeler applies `mk.linearize_product` in a single line.

### Effectiveness is Problem-Dependent

- **If the range of the integer is wide**, the auxiliary variables and constraints increase linearly. It is suited for products where the integer side is small, like batch count × batch size.
- It is only effective in cases where "the weakness of the non-convex relaxation is the bottleneck." In the [Diagnostic Benchmark](../../census.md), out of 11 nonlinear instances, `weak_relaxation` fired in only 1. That is why it is crucial to follow the procedure of confirming it with `mk.analyze` and adopting it **only after measuring its effectiveness** with `mk.compare_variants`.

Related: [Methods Guide 1. Exact Linearization of Integer × Continuous](../../playbook/01-linearize.md) /
API [`mk.linearize_product`](../../api/transforms.md)